In [6]:
import torch
import torch.nn as nn
import librosa
import numpy as np
import pyaudio
import torch.nn.functional as F

# Definir la clase del modelo LSTM
class EmotionRecognitionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(EmotionRecognitionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Definir la capa LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        
        # Definir la capa fully connected (FC)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # Inicializar los estados ocultos y las celdas de memoria para la LSTM
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # Pasar los datos a través de la capa LSTM
        out, _ = self.lstm(x, (h0, c0))
        
        # Tomar la salida de la última celda LSTM y pasarla por la capa fully connected
        out = self.fc(out[:, -1, :])
        
        return out

# Cargar el modelo guardado
input_size = 38  # Ajustamos a 38 características por timestep
hidden_size = 256
num_layers = 3
num_classes = 8  # Número de emociones

# Instanciar el modelo
model = EmotionRecognitionLSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, num_classes=num_classes)

# Cargar los pesos entrenados
model.load_state_dict(torch.load("final_model.pth"))

# Colocar el modelo en modo evaluación
model.eval()

# Función para extraer características de audio en tiempo real con 38 características por timestep
def extract_features_live(audio_data, sr, n_timesteps=100):
    if len(audio_data) == 0:
        return np.array([])

    hop_length = len(audio_data) // n_timesteps
    
    # Extraer características con un total de 38 características
    mfccs = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=13, hop_length=hop_length)  # 13 MFCC
    chroma = librosa.feature.chroma_stft(y=audio_data, sr=sr, hop_length=hop_length)     # 12 Chroma
    spec_contrast = librosa.feature.spectral_contrast(y=audio_data, sr=sr, hop_length=hop_length)  # 7 Spectral contrast
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(audio_data), sr=sr, hop_length=hop_length)  # 6 Tonnetz
    
    # Combinar características para cada timestep (13 + 12 + 7 + 6 = 38 características)
    features = np.hstack([mfccs.T, chroma.T, spec_contrast.T, tonnetz.T])

    return features

# Función para predecir emociones en tiempo real
def predict_emotion(audio_chunk, sr, model):
    # Extraer características
    features = extract_features_live(audio_chunk, sr, n_timesteps=100)
    
    if features.size == 0:
        return None

    # Redimensionar para coincidir con la entrada del modelo
    features = torch.tensor(features).float().unsqueeze(0)

    # Predicción
    with torch.no_grad():
        outputs = model(features)
        probabilities = F.softmax(outputs, dim=1)
        predicted_label = torch.argmax(probabilities, dim=1).item()

    return predicted_label

# Captura de audio en tiempo real con PyAudio
def record_audio_and_predict():
    chunk_size = 1024  # Tamaño del buffer
    sample_rate = 22050  # Frecuencia de muestreo
    format = pyaudio.paFloat32
    channels = 1  # Audio mono
    duration = 3  # Duración de la grabación

    # Inicializar PyAudio
    p = pyaudio.PyAudio()

    # Abrir el stream de audio
    stream = p.open(format=format, channels=channels, rate=sample_rate, input=True, frames_per_buffer=chunk_size)

    print("Iniciando predicción en tiempo real...")

    try:
        while True:
            # Leer datos de audio
            frames = []
            for _ in range(0, int(sample_rate / chunk_size * duration)):
                try:
                    data = stream.read(chunk_size)
                    frames.append(np.frombuffer(data, dtype=np.float32))
                except IOError as er:
                    if er.errno == pyaudio.paInputOverflowed:
                        print(f"Warning: Input overflowed, skipping this chunk")
                        data = '\x00' * chunk_size
    
            # Convertir los frames a un array de numpy
            audio_data = np.hstack(frames)

            # Predecir emoción
            predicted_emotion = predict_emotion(audio_data, sample_rate, model)

            if predicted_emotion is not None:
                print(f"Emoción predicha: {predicted_emotion}")
            else:
                print("No se pudo procesar el audio.")

    except KeyboardInterrupt:
        print("Interrupción manual.")

    # Detener y cerrar el stream
    stream.stop_stream()
    stream.close()
    p.terminate()

if __name__ == "__main__":
    record_audio_and_predict()


/var/folders/pv/hj1tkdtn4xd2lk842kp9xy7r0000gn/T/ipykernel_7806/271509836.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("final_model.

Iniciando predicción en tiempo real...
Emoción predicha: 5


ValueError: need at least one array to concatenate

In [1]:
import numpy as np
import librosa
import torch

# Función de extracción de características (igual a la que usaste para entrenar)
def extract_features_with_timesteps(file_name, n_timesteps=100):
    y, sr = librosa.load(file_name, sr=None)
    
    if len(y) == 0:
        print(f"Error: {file_name} no se pudo cargar o está vacío.")
        return np.array([])

    hop_length = len(y) // n_timesteps
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=hop_length)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=hop_length)
    spec_contrast = librosa.feature.spectral_contrast(y=y, sr=sr, hop_length=hop_length)
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr, hop_length=hop_length)
    
    features = np.hstack([mfccs.T, chroma.T, spec_contrast.T, tonnetz.T])
    
    return features

# Cargar el modelo previamente entrenado
model = torch.load('final_model.pth')
model.eval()  # Pon el modelo en modo evaluación

# Cargar y extraer características del archivo de prueba
audio_test_file = 'testData/Test_Audio.wav'
test_features = extract_features_with_timesteps(audio_test_file, n_timesteps=100)

if test_features.size > 0:
    # Preprocesar las características para alimentar al modelo
    test_features = test_features.reshape(1, test_features.shape[0], test_features.shape[1])  # Ajustar dimensiones
    
    # Convertir a tensor de PyTorch
    test_tensor = torch.tensor(test_features, dtype=torch.float32)
    
    # Hacer la predicción
    with torch.no_grad():
        prediction = model(test_tensor)
    
    # Mostrar el resultado
    predicted_class = torch.argmax(prediction, dim=1)
    print(f"Predicción para el archivo de prueba: {predicted_class.item()}")  # Aquí mapea la clase a la etiqueta correspondiente
else:
    print(f"No se pudieron extraer características de {audio_test_file}")


/var/folders/pv/hj1tkdtn4xd2lk842kp9xy7r0000gn/T/ipykernel_2565/4237144311.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('final_model.pth')


AttributeError: 'collections.OrderedDict' object has no attribute 'eval'